In [6]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

# --- ADDED: Load the features from the CSV we generated ---
print('Loading features from disk...')
features_df = pd.read_csv('data/features.csv')
# ----------------------------------------------------------

# 1. Patient-Level Group Split (Strict Leakage Prevention)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(
    gss.split(features_df, features_df['label'], groups=features_df['record_id'])
)

train_df = features_df.iloc[train_idx].copy()
test_df = features_df.iloc[test_idx].copy()

feature_cols = [
    'mean_amplitude',
    'std_amplitude',
    'max_amplitude',
    'min_amplitude',
    'peak_to_peak',
    'rr_interval',
]

X_train, y_train = train_df[feature_cols], train_df['label']
X_test, y_test = test_df[feature_cols], test_df['label']

# Calculate negative-to-positive ratio for XGBoost
scale_pos_weight_val = (y_train == 0).sum() / (y_train == 1).sum()

# 2. Model Zoo Configuration
models = {
    'Logistic Regression': Pipeline(
        [
            ('scaler', StandardScaler()),
            (
                'clf',
                LogisticRegression(
                    class_weight='balanced', max_iter=1000, random_state=42
                ),
            ),
        ]
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42,
    ),
    'Support Vector Machine (RBF)': Pipeline(
        [
            ('scaler', StandardScaler()),
            (
                'clf',
                SVC(
                    kernel='rbf',
                    C=1.0,
                    class_weight='balanced',
                    probability=True,
                    random_state=42,
                ),
            ),
        ]
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        scale_pos_weight=scale_pos_weight_val,
        eval_metric='logloss',
        n_jobs=-1,
        random_state=42,
    ),
}

# 3. Training and Evaluation Loop
results = []

for name, model in models.items():
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  y_prob = (
      model.predict_proba(X_test)[:, 1]
      if hasattr(model, 'predict_proba')
      else None
  )

  results.append({
      'Model': name,
      'Accuracy': accuracy_score(y_test, y_pred),
      'Abnormal Recall': recall_score(y_test, y_pred, pos_label=1),
      'Abnormal Precision': precision_score(y_test, y_pred, pos_label=1),
      'Macro F1': f1_score(y_test, y_pred, average='macro'),
      'ROC-AUC': roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan,
  })

# 4. Comparative Benchmark Summary
benchmark_df = (
    pd.DataFrame(results).sort_values(by='Macro F1', ascending=False).reset_index(drop=True)
)
print(benchmark_df.to_string(index=False))


Loading features from disk...


/Users/sherif_elgendy/Documents/Ecg/ECG_Arrhythmia_Detection/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[LightGBM] [Info] Number of positive: 24235, number of negative: 58985
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000477 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 83220, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
                       Model  Accuracy  Abnormal Recall  Abnormal Precision  Macro F1  ROC-AUC
         Logistic Regression  0.721417         0.671121            0.504633  0.684315 0.814401
Support Vector Machine (RBF)  0.635397         0.594952            0.401302  0.599397 0.725594
                    LightGBM  0.624759         0.626726            0.395706  0.594967 0.697289
                     XGBoost  0.616760         0.624053            0.388376  0.587880 0.703194
               Random Forest  0.